# Detect · Plan · Grasp — Day 2: Train the YCB detector

Self-contained (no repo clone needed). **Runtime → Change runtime type → GPU**, then **Run all**.

- Trains **YOLOv8-nano** on synthetic YCB frames (`train_pbr`)
- Validates on **real** frames (`test_bop19`) — an honest sim→real check
- Exports **ONNX** (opset 13) for Day 3, and packages weights + curves for download

Mirrors `src/prepare_ycb.py` + `src/train.py` from the repo.

In [ ]:
!nvidia-smi -L

In [ ]:
!pip -q install ultralytics huggingface_hub

### Knobs — reduce if a free Colab session times out

In [ ]:
N_TRAIN_SCENES = 15   # synthetic pbr scenes to use (~1000 imgs each)
EPOCHS = 50
BATCH  = 64           # fits T4/A100; lower if you hit OOM
MIN_VISIB = 0.1

### 1 · Download data from HuggingFace (BOP ycbv)

In [ ]:
import os, zipfile, glob
os.makedirs('data', exist_ok=True)
HF = 'https://huggingface.co/datasets/bop-benchmark/ycbv/resolve/main'

# val: real test frames (630 MB)
!wget -q --show-progress -O data/test.zip {HF}/ycbv_test_bop19.zip
with zipfile.ZipFile('data/test.zip') as z: z.extractall('data')

# train: synthetic pbr (~20 GB). Download once, extract only N scenes to save disk/time.
!wget -q --show-progress -O data/pbr.zip {HF}/ycbv_train_pbr.zip
keep = {f'{i:06d}' for i in range(N_TRAIN_SCENES)}
with zipfile.ZipFile('data/pbr.zip') as z:
    members = [n for n in z.namelist() if any(('/'+s+'/') in ('/'+n) for s in keep)]
    z.extractall('data', members)
!rm -f data/test.zip data/pbr.zip

# locate the extracted split roots (top folder name can vary)
gts = glob.glob('data/**/scene_gt.json', recursive=True)
TEST_ROOT = next(p for p in gts if 'pbr' not in p).rsplit('/',2)[0]
PBR_ROOT  = next(p for p in gts if 'pbr' in p).rsplit('/',2)[0]
print('TEST_ROOT =', TEST_ROOT, '| PBR_ROOT =', PBR_ROOT)

### 2 · Convert BOP → YOLO  *(inline copy of `src/prepare_ycb.py`)*

In [ ]:
import json
from pathlib import Path

YCB_CLASSES = [
 "002_master_chef_can","003_cracker_box","004_sugar_box","005_tomato_soup_can",
 "006_mustard_bottle","007_tuna_fish_can","008_pudding_box","009_gelatin_box",
 "010_potted_meat_can","011_banana","019_pitcher_base","021_bleach_cleanser",
 "024_bowl","025_mug","035_power_drill","036_wood_block","037_scissors",
 "040_large_marker","051_large_clamp","052_extra_large_clamp","061_foam_brick"]
IMG_W, IMG_H = 640, 480

def convert_split(split_dir, out_root, name, min_visib=MIN_VISIB):
    split_dir, out_root = Path(split_dir), Path(out_root)
    img_dir = out_root/"images"/name; lbl_dir = out_root/"labels"/name
    img_dir.mkdir(parents=True, exist_ok=True); lbl_dir.mkdir(parents=True, exist_ok=True)
    n_img=n_box=n_drop=0
    for scene in sorted(p for p in split_dir.iterdir() if p.is_dir()):
        gt   = json.loads((scene/"scene_gt.json").read_text())
        info = json.loads((scene/"scene_gt_info.json").read_text())
        for img_id, objs in gt.items():
            lines=[]
            for obj, meta in zip(objs, info[img_id]):
                if meta["visib_fract"] < min_visib: n_drop+=1; continue
                x,y,w,h = meta["bbox_visib"]
                if w<=0 or h<=0: n_drop+=1; continue
                cls = obj["obj_id"]-1
                lines.append(f"{cls} {(x+w/2)/IMG_W:.6f} {(y+h/2)/IMG_H:.6f} {w/IMG_W:.6f} {h/IMG_H:.6f}")
                n_box+=1
            stem=f"{scene.name}_{int(img_id):06d}"
            (lbl_dir/f"{stem}.txt").write_text("\n".join(lines))
            src=next((scene/"rgb").glob(f"{int(img_id):06d}.*"))
            dst=img_dir/f"{stem}{src.suffix}"
            if dst.exists() or dst.is_symlink(): dst.unlink()
            dst.symlink_to(src.resolve())
            n_img+=1
    print(f"[{name}] {n_img} imgs, {n_box} boxes ({n_drop} dropped)")

def write_yaml(out_root):
    names="\n".join(f"  {i}: {n}" for i,n in enumerate(YCB_CLASSES))
    Path(out_root,"data.yaml").write_text(
        f"path: {Path(out_root).resolve()}\ntrain: images/train\nval: images/val\n"
        f"nc: {len(YCB_CLASSES)}\nnames:\n{names}\n")

convert_split(TEST_ROOT, "data/ycb_yolo", "val")
convert_split(PBR_ROOT,  "data/ycb_yolo", "train")
write_yaml("data/ycb_yolo")
print(open("data/ycb_yolo/data.yaml").read()[:200])

### 3 · Train YOLOv8-nano (transfer from COCO)

In [ ]:
from ultralytics import YOLO
model = YOLO("yolov8n.pt")
model.train(data="data/ycb_yolo/data.yaml", epochs=EPOCHS, imgsz=640,
            batch=BATCH, device=0, project="runs", name="ycb_detector", patience=15)

### 4 · Validate on real frames + export ONNX + package (robust to run path)

In [ ]:
import glob, os, shutil
from google.colab import files as gf
from ultralytics import YOLO

# Ultralytics may log to runs/detect/... — glob for best.pt wherever it landed.
best = glob.glob("/content/**/ycb_detector*/weights/best.pt", recursive=True)[0]
run  = os.path.dirname(os.path.dirname(best))
print("run dir:", run)

m = YOLO(best).val(data="data/ycb_yolo/data.yaml")
print(f"val mAP50 = {m.box.map50:.4f} | mAP50-95 = {m.box.map:.4f}")
YOLO(best).export(format="onnx", opset=13, imgsz=640)

os.makedirs("/content/artifacts", exist_ok=True)
for pat in ["weights/best.pt","weights/best.onnx","results.csv","results.png",
            "confusion_matrix.png","*PR_curve.png","val_batch0_pred.jpg","args.yaml"]:
    for f in glob.glob(f"{run}/{pat}"):
        shutil.copy(f, "/content/artifacts/")
shutil.make_archive("/content/ycb_detector_artifacts", "zip", "/content/artifacts")
gf.download("/content/ycb_detector_artifacts.zip")